Config and metadata

In [0]:
import requests, json, time
from pathlib import Path

API  = "https://statistikdatabasen.scb.se/api/v2"
ROOT = Path("/Volumes/laddstolpar_df/landing/raw/scb")
TABLES = ["TAB3277", "TAB628", "TAB3276"]

cfg = requests.get(f"{API}/config", timeout=30).json()
MAX_CELLS = cfg["maxDataCells"]
print("maxDataCells:", MAX_CELLS)

def get_json(path):
    r = requests.get(f"{API}/{path}", params={"lang": "sv"}, timeout=30)
    r.raise_for_status()
    return r.json()

def codes(meta, var_id):
    idx = meta["dimension"][var_id]["category"]["index"]
    return list(idx.keys()) if isinstance(idx, dict) else list(idx)

for tab in TABLES:
    info = get_json(f"tables/{tab}")
    meta = get_json(f"tables/{tab}/metadata")
    print(f"\n{tab}  updated={info.get('updated')}  periods={info.get('firstPeriod')}–{info.get('lastPeriod')}")
    total = 1
    for v in meta["id"]:
        c = codes(meta, v)
        total *= len(c)
        print(f"  {v:<15} {len(c):>4} values   e.g. {c[:4]} … {c[-2:]}")
    print(f"  full table = {total:,} cells")

Loader - defs

In [0]:
import re

PAUSE_S = 0.4          # SCB: max 30 calls / 10 s (Labb2)

def safe(ts: str) -> str:
    """'2026-09-02T06:00:00Z' -> '20260902T060000Z' (':' breaks Spark paths)."""
    return re.sub(r"[^0-9A-Za-z]", "", ts)

def post_data(tab, selection, tries=4):
    body = {"selection": [{"variableCode": k, "valueCodes": v} for k, v in selection.items()]}
    for attempt in range(1, tries + 1):
        r = requests.post(f"{API}/tables/{tab}/data",
                          params={"lang": "sv", "outputFormat": "json-stat2"},
                          json=body, timeout=120)
        if r.status_code == 200:
            return r
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        raise RuntimeError(f"{tab}: HTTP {r.status_code} {r.text[:300]}")   # 400 = our error, no retry
    r.raise_for_status()

def land_scb(tab, fixed, page_var, pages):
    """fixed: {variable: [codes]} for the non-paged variables.
       pages: {page_name: [codes of page_var]} – one API call per page."""
    info = get_json(f"tables/{tab}")
    folder = ROOT / tab / f"updated={safe(info['updated'])}"     # new table version -> new folder
    stats = {"table": tab, "updated": info["updated"], "calls": 0, "written": 0, "skipped": 0, "cells": 0}
    for name, page_codes in pages.items():
        p = folder / f"{page_var}={name}.json"
        if p.exists() and p.stat().st_size > 0:                   # idempotency
            stats["skipped"] += 1
            continue
        sel = {**fixed, page_var: page_codes}
        expected = 1
        for v in sel.values():
            expected *= len(v)
        assert expected <= MAX_CELLS, f"{tab}/{name}: {expected:,} cells > {MAX_CELLS:,}"
        r = post_data(tab, sel)
        got = len(r.json()["value"])
        assert got == expected, f"{tab}/{name}: got {got:,} cells, expected {expected:,}"
        folder.mkdir(parents=True, exist_ok=True)
        p.write_text(r.text, encoding="utf-8")                    # raw response, unchanged
        stats["calls"] += 1; stats["written"] += 1; stats["cells"] += got
        time.sleep(PAUSE_S)
    return stats

3 check tables

In [0]:
m3277 = get_json("tables/TAB3277/metadata")
m628  = get_json("tables/TAB628/metadata")
m3276 = get_json("tables/TAB3276/metadata")

YEARS  = [str(y) for y in range(2020, 2026)]                    # 2020–2025
months = [t for t in codes(m3277, "Tid") if t >= "2021M01"]      # 2021M01 → latest
month_pages = {y: [t for t in months if t.startswith(y)] for y in sorted({t[:4] for t in months})}

plans = [
    ("TAB628",  {"Region": codes(m628, "Region"), "Kon": ["1+2"],
                 "ContentsCode": codes(m628, "ContentsCode")},            "Tid", {"2020-2025": YEARS}),
    ("TAB3276", {"Region": codes(m3276, "Region"), "Agarkategori": codes(m3276, "Agarkategori"),
                 "ContentsCode": codes(m3276, "ContentsCode")},           "Tid", {"2020-2025": YEARS}),
    ("TAB3277", {"Region": codes(m3277, "Region"), "Drivmedel": codes(m3277, "Drivmedel"),
                 "ContentsCode": codes(m3277, "ContentsCode")},           "Tid", month_pages),
]
for tab, fixed, page_var, pages in plans:
    print(land_scb(tab, fixed, page_var, pages))

extra = sorted(set(codes(m3277, "Region")) - set(codes(m628, "Region")))
labels = m3277["dimension"]["Region"]["category"]["label"]
print("Region codes in TAB3277 but not TAB628:", [(c, labels[c]) for c in extra])

flatten and load bronze - defs

In [0]:
import itertools
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

def flatten_jsonstat2(js):
    """json-stat2 is row-major (last dimension varies fastest) -> itertools.product order."""
    cats = []
    for v in js["id"]:
        idx = js["dimension"][v]["category"]["index"]
        cats.append(sorted(idx, key=idx.get) if isinstance(idx, dict) else list(idx))
    values, status = js["value"], js.get("status") or {}
    for i, combo in enumerate(itertools.product(*cats)):
        st = status.get(str(i)) if isinstance(status, dict) else (status[i] if status else None)
        v = values[i]
        yield (*combo, float(v) if v is not None else None, st)   # float: DoubleType rejects int

def bronze_scb(tab):
    target = f"laddstolpar_df.bronze.scb_{tab.lower()}"
    done = set()
    if spark.catalog.tableExists(target):                         # the processed-files ledger
        done = {r[0] for r in spark.table(target).select("_file").distinct().collect()}
    files = sorted(str(p) for p in (ROOT / tab).rglob("*.json"))
    stats = {"table": target, "files_total": len(files), "new_files": 0, "rows_written": 0}
    for f in files:
        if f in done:
            continue
        js = json.loads(Path(f).read_text(encoding="utf-8"))
        cols = [v.lower() for v in js["id"]]                      # region, drivmedel, contentscode, tid …
        schema = StructType([StructField(c, StringType()) for c in cols]     # codes stay strings
                            + [StructField("value", DoubleType()),
                               StructField("status", StringType())])         # '..' markers etc.
        rows = list(flatten_jsonstat2(js))
        assert len(rows) == len(js["value"]), f"{f}: flatten mismatch"
        df = (spark.createDataFrame(rows, schema)
                .withColumn("_snapshot_updated", F.lit(re.search(r"updated=(\w+)", f).group(1)))
                .withColumn("_file", F.lit(f))
                .withColumn("_ingested_at", F.current_timestamp()))
        df.write.mode("append").saveAsTable(target)               # one commit per file
        stats["new_files"] += 1
        stats["rows_written"] += len(rows)
    return stats

Run loader

In [0]:
for tab in ["TAB628", "TAB3276", "TAB3277"]:
    print(bronze_scb(tab))

6 Checks tables 

In [0]:
%sql
SELECT 'scb_tab628' AS t, COUNT(*) AS rows_, COUNT(DISTINCT _file) AS files,
       COUNT_IF(value IS NULL) AS null_values, COUNT_IF(status IS NOT NULL) AS flagged
FROM laddstolpar_df.bronze.scb_tab628
UNION ALL
SELECT 'scb_tab3276', COUNT(*), COUNT(DISTINCT _file), COUNT_IF(value IS NULL), COUNT_IF(status IS NOT NULL)
FROM laddstolpar_df.bronze.scb_tab3276
UNION ALL
SELECT 'scb_tab3277', COUNT(*), COUNT(DISTINCT _file), COUNT_IF(value IS NULL), COUNT_IF(status IS NOT NULL)
FROM laddstolpar_df.bronze.scb_tab3277;

-- Heby: old code 1917 vs current 0331 in our period
SELECT region, COUNT(*) AS rows_, SUM(value) AS total, COUNT_IF(value IS NULL) AS nulls
FROM laddstolpar_df.bronze.scb_tab3277
WHERE region IN ('1917', '0331')
GROUP BY region;

extra checks

In [0]:
%sql
-- Which owner category, year and flag?
SELECT agarkategori, tid, status, COUNT(*) AS n
FROM laddstolpar_df.bronze.scb_tab3276
WHERE value IS NULL
GROUP BY ALL
ORDER BY n DESC;

-- Which regions escape it (if 314 = 315 − 1)?
SELECT region, COUNT_IF(value IS NULL) AS nulls, COUNT(*) AS rows_
FROM laddstolpar_df.bronze.scb_tab3276
GROUP BY region
HAVING nulls = 0 OR nulls > 1
ORDER BY nulls
LIMIT 20;

In [0]:
lab = m3276["dimension"]["Agarkategori"]["category"]["label"]
print("Ägarkategori:", lab)

In [0]:
%sql
SELECT region, COUNT(*) AS nulls
FROM laddstolpar_df.bronze.scb_tab3276
WHERE value IS NULL AND agarkategori <> '040'
GROUP BY region;
-- expect exactly 15, 16, 1917 with 36 each